In [14]:
import numpy as np
import pandas as pd
import yfinance as yf                                     # 야후 파이낸스 주가 다운로드
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier  #금융 표 데이터 표준 베이스라인
from sklearn.inspection import permutation_importance

In [15]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "AMD",
           "AVGO", "QCOM", "INTC", "ORCL", "CSCO", "ADBE", "NFLX"]   # 15종목

frames = []

for ticker in tickers:
    d= yf.download(ticker, start="2018-01-01", end = "2026-07-01", 
                   auto_adjust=True, progress=False)
    d.columns = d.columns.get_level_values(0)
    d["Ticker"] = ticker
    frames.append(d)

df = pd.concat(frames).reset_index().set_index(["Date", "Ticker"]).sort_index()
print(df.shape)
df.head()


(32010, 5)


Price                   Close        High         Low        Open     Volume
Date       Ticker                                                           
2018-01-02 AAPL     40.232384   40.241728   39.531715   39.741918  102223600
           ADBE    177.699997  177.800003  175.259995  175.850006    2432800
           AMD      10.980000   11.020000   10.340000   10.420000   44146300
           AMZN     59.450500   59.500000   58.525501   58.599998   53890000
           AVGO     21.015076   21.053641   20.272098   20.445249   33135000

In [16]:
g = lambda: df.groupby(level="Ticker")

df["ret_1"] = g()["Close"].pct_change()
df["ret_5"] = g()["Close"].pct_change(5)
df["vol_20"] = g()["ret_1"].transform(lambda s : s.rolling(20).std())
df["hl_spread"] = (df["High"] - df["Low"]) / df["Close"]  # 당일 고저폭
df["vol_z20"]   = g()["Volume"].transform(                # 거래량이 평소보다 얼마나 튀었나
    lambda s: (s - s.rolling(20).mean()) / s.rolling(20).std())

FEATURES = ["ret_5", "vol_20", "hl_spread", "vol_z20"]

In [17]:
df["fwd_ret_5"] = g()["Close"].shift(-5) / df["Close"] -1

df["y"] = (df.groupby(level="Date")["fwd_ret_5"].rank(pct=True) > 0.7).astype(int)

df = df.dropna(subset=FEATURES + ["fwd_ret_5"])              # 계산 안 되는 앞뒤 날짜 제거
print(df["y"].mean())                                       

0.3333333333333333


In [18]:
dates = df.index.get_level_values("Date").unique().sort_values()
test_starts = ["2024-01-01", "2024-07-01", "2025-01-01", "2025-07-01"]   # 반기씩 4폴드
test_ends   = ["2024-06-30", "2024-12-31", "2025-06-30", "2025-12-31"]
PURGE = 5                                                    # 라벨이 5일 미래를 보니까 5일 잘라냄

def rank_ic(d):                                              # 하루 단위 RankIC
    return d["pred"].corr(d["fwd_ret_5"], method="spearman")

models = {
    "RF":   lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50,
                                           n_jobs=-1, random_state=0),
    "LGBM": lambda: LGBMClassifier(n_estimators=300, learning_rate=0.02, num_leaves=15,
                                   min_child_samples=100, verbose=-1, random_state=0),
}
def perm_importance_within_date(model, te, features, n_repeats=5, seed=0):
    rng = np.random.default_rng(seed)
    base = te.groupby(level="Date").apply(rank_ic).mean()        # 섞기 전 RankIC
    out = {}
    for f in features:
        drops = []
        for _ in range(n_repeats):
            X = te[features].copy()
            X[f] = X.groupby(level="Date")[f].transform(         # 같은 날짜 안에서만 섞기
                lambda s: rng.permutation(s.values))
            tmp = te.copy()
            tmp["pred"] = model.predict_proba(X)[:, 1]
            drops.append(base - tmp.groupby(level="Date").apply(rank_ic).mean())
        out[f] = round(float(np.mean(drops)), 4)
    return out

for name, make in models.items():
    for s, e in zip(test_starts, test_ends):
        s_idx = dates.searchsorted(pd.Timestamp(s))
        train_end = dates[s_idx - PURGE]

        D  = df.index.get_level_values("Date")
        tr = df[D < train_end]
        te = df[(D >= pd.Timestamp(s)) & (D <= pd.Timestamp(e))].copy()

        model = make().fit(tr[FEATURES], tr["y"])                # ① 학습
        te["pred"] = model.predict_proba(te[FEATURES])[:, 1]     # ② 예측

        ic = te.groupby(level="Date").apply(rank_ic)
        print(f"{name:<4} {s[:7]}  AUC={roc_auc_score(te['y'], te['pred']):.3f}  "
              f"RankIC={ic.mean():+.3f}  양수일={(ic > 0).mean():.0%}")
        print("   ", perm_importance_within_date(model, te, FEATURES))   # ③ 그다음 중요도

RF   2024-01  AUC=0.553  RankIC=+0.034  양수일=56%
    {'ret_5': -0.0016, 'vol_20': 0.0458, 'hl_spread': 0.0078, 'vol_z20': -0.0123}
RF   2024-07  AUC=0.496  RankIC=-0.050  양수일=41%
    {'ret_5': -0.0003, 'vol_20': -0.012, 'hl_spread': 0.015, 'vol_z20': -0.0217}
RF   2025-01  AUC=0.512  RankIC=+0.010  양수일=51%
    {'ret_5': 0.0101, 'vol_20': 0.0198, 'hl_spread': -0.0223, 'vol_z20': -0.0019}
RF   2025-07  AUC=0.541  RankIC=+0.028  양수일=54%
    {'ret_5': -0.0041, 'vol_20': 0.0341, 'hl_spread': 0.0002, 'vol_z20': -0.0085}
LGBM 2024-01  AUC=0.565  RankIC=+0.056  양수일=56%
    {'ret_5': 0.0056, 'vol_20': 0.0801, 'hl_spread': 0.0242, 'vol_z20': -0.001}
LGBM 2024-07  AUC=0.492  RankIC=-0.061  양수일=46%
    {'ret_5': -0.0022, 'vol_20': -0.0282, 'hl_spread': 0.0076, 'vol_z20': 0.0059}
LGBM 2025-01  AUC=0.518  RankIC=+0.017  양수일=53%
    {'ret_5': 0.0244, 'vol_20': 0.0062, 'hl_spread': -0.0099, 'vol_z20': -0.0105}
LGBM 2025-07  AUC=0.551  RankIC=+0.032  양수일=55%
    {'ret_5': 0.0026, 'vol_20': 0.0372, 'hl_s

In [19]:
for s, e in zip(test_starts, test_ends):
    D = df.index.get_level_values("Date")
    seg = df[(D >= pd.Timestamp(s)) & (D <= pd.Timestamp(e))]
    fic = seg.groupby(level="Date").apply(
        lambda d: d["vol_20"].corr(d["fwd_ret_5"], method="spearman"))
    print(f"{s[:7]}  vol_20 IC={fic.mean():+.3f}")

2024-01  vol_20 IC=+0.036
2024-07  vol_20 IC=-0.088
2025-01  vol_20 IC=-0.022
2025-07  vol_20 IC=+0.013
